# Transfer Learning — AI for Bean Leaf Health
**PyTorch Community Meetup Mbarara · 25 minutes**

Goal: adapt pretrained ResNet18 to classify healthy bean leaves, bean rust, and angular leaf spot. Research demonstration only: not a validated diagnostic tool.

**Before the session:** run all cells once and save the notebook outputs. The pretrained weights download requires internet.

**Next steps**

- Open the notebook in Colab or Jupyter and make sure the runtime can download the dataset and model weights.

## 1. Check your environment (1 minute)
Install the Beans workshop dependencies using the repository README. In Colab, run `%pip install datasets` in a new cell before continuing.

**Next steps**

- After installing the dependencies, run `import datasets` in a new cell to check that the active kernel can find the package.

## 2. Load a Uganda-relevant dataset (3 minutes)
Makerere AI Lab Beans dataset: https://huggingface.co/datasets/AI-Lab-Makerere/beans

**Next steps**

- Run the dataset-loading cell and identify the training and validation splits before fitting the model.

In [ ]:
import torch, matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

raw = load_dataset("AI-Lab-Makerere/beans")
print(raw)
class_names = raw["train"].features["labels"].names
print("Classes:", class_names)

**Next steps**

- Run `print(len(raw['train']), len(raw['validation']))` and write down the two split sizes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
seen = set()
for item in raw["train"]:
    label = item["labels"]
    if label not in seen:
        ax = axes[len(seen)]
        ax.imshow(item["image"].convert("RGB"))
        ax.set_title(class_names[label])
        ax.axis("off")
        seen.add(label)
    if len(seen) == 3: break
plt.tight_layout()

**Next steps**

- Choose one visible cue from the example leaves and check whether it appears in more than one class.

## 3. Preprocess images (3 minutes)
Resize and normalize with the transforms associated with ResNet18 pretrained weights.

**Next steps**

- Run the preprocessing cell, then compare a raw image's size with the shape of its preprocessed tensor.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

class BeanImages(Dataset):
    def __init__(self, hf_split): self.items = hf_split
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        item = self.items[i]
        return preprocess(item["image"].convert("RGB")), int(item["labels"])

# Restrict training examples to keep live training short. Validation remains independent.
gen = torch.Generator().manual_seed(42)
indices = torch.randperm(len(raw["train"]), generator=gen)[:300].tolist()
train_ds = BeanImages(raw["train"].select(indices))
valid_ds = BeanImages(raw["validation"])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=32, num_workers=0)
print(len(train_ds), "training images;", len(valid_ds), "validation images")

**Next steps**

- Run `print(train_ds[0][0].shape, train_ds[0][1])` and use `class_names` to look up the label.

## 4. Load pretrained model and replace its head (3 minutes)
Freeze the ResNet backbone; learn only the 3-way linear classifier.

**Next steps**

- Predict how many output scores the new head should produce for one leaf, then run the model-loading cell.

In [ ]:
from torch import nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)
print("Device:",device)
print("Trainable parameters:",sum(p.numel() for p in model.parameters() if p.requires_grad))

**Next steps**

- Run `print([name for name, p in model.named_parameters() if p.requires_grad])` to verify that only the classifier head can learn.

## 5. Train the classification head (6 minutes)
We put the backbone in evaluation mode so its BatchNorm statistics stay fixed while the new head learns.

**Next steps**

- Run the training cell and keep its printed loss as your one-epoch baseline.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

def train_epoch():
    model.eval()  # keeps pretrained batch normalization statistics frozen
    model.fc.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        scores = model(x)
        loss = criterion(scores, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(train_ds)

# One epoch is the live version; increase to 3-5 before the talk if time allows.
print("Training loss:",round(train_epoch(),4))

**Next steps**

- Record the training loss, then run the validation cell to measure performance on separate images.

## 6. Evaluate on held-out validation examples (3 minutes)

**Next steps**

- Before evaluating, predict which leaf class will be hardest to identify. Revisit that prediction after viewing the examples.

In [ ]:
model.eval()
correct, total = 0, 0
with torch.inference_mode():
    for x,y in valid_loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        correct += (pred == y).sum().item()
        total += len(y)
print(f"Validation accuracy: {correct / total:.1%} ({correct}/{total})")

**Next steps**

- Record the validation accuracy and compute `total - correct` to count validation mistakes.

## 7. See predictions on actual bean leaves (2 minutes)

**Next steps**

- Run the prediction gallery and compare each predicted label with the true label.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
model.eval()
for i,ax in enumerate(axes.flat):
    sample = raw["validation"][i]
    x = preprocess(sample["image"].convert("RGB")).unsqueeze(0).to(device)
    with torch.inference_mode():
        prediction = model(x).argmax(dim=1).item()
    ax.imshow(sample["image"].convert("RGB"))
    ax.set_title(f"Pred: {class_names[prediction]}\nTrue: {class_names[sample['labels']]}",fontsize=9)
    ax.axis("off")
plt.tight_layout()

**Next steps**

- Change `raw['validation'][i]` to `raw['validation'][i + 6]`, rerun the gallery, and compare the next six predictions.

## 8. Take-home challenge and implications (4 minutes)
- Increase training data from 300 examples to the full training split.
- Train for 3 epochs and compare validation accuracy.
- Unfreeze only the final ResNet block and compare results. Rebuild the optimizer with the newly trainable parameters and use a smaller learning rate.
- Test whether varied lighting and camera quality affect predictions.

**Important:** This is a classroom demonstration. A field-ready plant disease model requires representative data, external validation, and agronomic expertise.

**Connection to FastDRS:** The same workflow (pretrained visual representations, adaptation, validation) appears in medical image research, but medical use requires substantially stricter safeguards.

**Next steps**

- Choose one experiment, such as training for three epochs. Rerun from a fresh model and compare its validation accuracy with the one-epoch baseline.